# 第 2 章 · block 为什么必须互不依赖：block 怎么被分到 SM 上

本章回答：一个 grid 有 1000 个 block，GPU 只有 132 个 SM，它们是怎么被执行的？为什么 CUDA 规定不同 block 之间不能互相等？"一波一波"执行会带来什么性能问题？

配套讲义：仓库里 `cuda-programming/notes/lessons/02-block-scheduling.md`（讲解、动画、练习都在讲义里）。对应原文：PG §1.2.2.1（`materials/text/cuda-programming-guide.txt` 第 1000–1058 行）、§2.3.7 Kernel Launch and Occupancy（第 4431–4470 行）、§3.2.2.2 Hardware Multithreading（第 8716–8753 行）；Oxford lec1 第 19 页（`text/oxford-lec1.txt` 第 278–292 行）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import math

import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一、二步 调度顺序不保证 | 谁空派给谁 | `schedule(10, 2, 2, seed)` 打乱完成顺序，`run_blocks` 每个 block 只写自己的部分和 | 三种顺序结果相同 |
| 第三步 4 个 SM × 2 | $\lceil 20 / 8 \rceil$ | `wave_stats(20, 4, 2)` | 3 波，最后一波 4 个，利用率 83% |
| 第三步 H100 表 | $133 / 264$ | `wave_stats(133, 132, 1)` | 2 波，最后一波 1 / 132，50.4% |
| 第三步 block 太少 | 132 − 50 | `132 - 50` | 82 个 SM 闲着 |
| 第四步 切 warp | 线程号 ÷ 32 | `warp_of(t)` | 0、31 → warp 0；32、63 → warp 1 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `run_blocks`**（来自 `ch02_block_waves.py`）：每个 block 只写自己那一段的部分和（不读别的 block 的结果），按给定顺序执行。

In [ ]:
def run_blocks(data: np.ndarray, block_dim: int, order: list[int]) -> np.ndarray:
    """每个 block 只写自己那一段的部分和（不读别的 block 的结果），按给定顺序执行。"""
    out = np.zeros(math.ceil(len(data) / block_dim))
    for b in order:
        out[b] = data[b * block_dim:(b + 1) * block_dim].sum()
    return out

**函数 `schedule`**（来自 `ch02_block_waves.py`）：模拟调度器：每一波把还没做的 block 派到空槽位上；同一波里谁先做完是随机的（seed 决定）。

In [ ]:
def schedule(grid: int, sms: int, per_sm: int, seed: int) -> list[list[int]]:
    """模拟调度器：每一波把还没做的 block 派到空槽位上；同一波里谁先做完是随机的（seed 决定）。

    返回每一波实际派出去的 block 编号。
    """
    rng = np.random.default_rng(seed)
    todo = list(range(grid))
    waves = []
    while todo:
        slots = sms * per_sm
        batch, todo = todo[:slots], todo[slots:]
        waves.append([int(x) for x in rng.permutation(batch)])  # 完成顺序被打乱
    return waves

**函数 `warp_of`**（来自 `ch02_block_waves.py`）：block 里第 thread_idx 号线程属于第几个 warp（每 32 个一组，切法固定）。

In [ ]:
def warp_of(thread_idx: int) -> int:
    """block 里第 thread_idx 号线程属于第几个 warp（每 32 个一组，切法固定）。"""
    return thread_idx // 32

**函数 `wave_stats`**（来自 `ch02_block_waves.py`）：波数 = ⌈grid / (SM 数 × 每 SM 的 block 数)⌉；返回（波数, 最后一波的 block 数, 利用率）。

In [ ]:
def wave_stats(grid: int, sms: int, per_sm: int) -> tuple[int, int, float]:
    """波数 = ⌈grid / (SM 数 × 每 SM 的 block 数)⌉；返回（波数, 最后一波的 block 数, 利用率）。"""
    per_wave = sms * per_sm
    waves = math.ceil(grid / per_wave)
    last = grid - (waves - 1) * per_wave
    return waves, last, grid / (waves * per_wave)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 block 之间不互相等：换三种完成顺序，结果都一样")
data = np.arange(80, dtype=np.float64)
results = []
for seed in range(3):
    order = [b for wave in schedule(10, 2, 2, seed) for b in wave]
    results.append(run_blocks(data, 8, order))
    print(f"  seed {seed} 的执行顺序 {order} → 部分和 {results[-1][:3]}…")
print("  三次结果相同：", all(np.array_equal(results[0], r) for r in results))

In [ ]:
print("\n第三步 4 个 SM、每个放 2 个 block、grid = 20")
print("  每一波派出：", [len(w) for w in schedule(20, 4, 2, 0)])
waves, last, util = wave_stats(20, 4, 2)
print(f"  波数 = {waves}，最后一波 {last} 个，利用率 = {util:.0%}")

In [ ]:
print("\n第三步 H100：132 个 SM，每个放 1 个 block")
for grid in (132, 133, 264):
    waves, last, util = wave_stats(grid, 132, 1)
    print(f"  grid {grid:3d}: 波数 {waves}，最后一波 {last:3d} / 132，利用率 {util:.1%}")
print("  grid 只有 50 个时，从头闲到尾的 SM 数 =", 132 - 50)

In [ ]:
print("\n第四步 block 切成 warp：线程号 → warp 号")
for t in (0, 31, 32, 63):
    print(f"  线程 {t:2d} → warp {warp_of(t)}")

## 3. 代码里才看得到的

1. **结果和顺序无关，是因为 block 只写自己那一格**：`run_blocks` 里 block b 只写 `out[b]`、只读自己那一段数据。要是某个 block 去读别人的 `out`，换个 seed 结果就会变——这正是第二步说的"block 之间不许等"。
2. **`wave_stats` 假设每个 block 耗时一样**：真实 GPU 上 block 耗时有快有慢，调度器会把新 block 派给先空出来的 SM，所以"波"的边界没有这么整齐；但"最后一段时间有 SM 闲着"这件事不变。

## 4. 自己改一改

- 把 `wave_stats(20, 4, 2)` 的 grid 换成 33（动画里的另一个开关），看最后一波有多空。
- 在 `run_blocks` 里故意让 block b 读 `out[b - 1]`，换几个 seed 看结果还一不一样。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> block 是工单，SM 是车间，谁空派给谁，顺序不保证；
> 派了就做完，中途不换车间；
> block 之间不许等，要交换就拆 kernel、用原子、上 cluster；
> 一次跑满叫一波，最后一波不满是尾巴，block 数要是 SM 数的好几倍。